In [53]:
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer
from textblob import TextBlob
from transformers import AutoTokenizer, AutoModelForSequenceClassification, pipeline
import torch
import numpy as np
import torch.nn.functional as F
import re


In [21]:
# Load model and tokenizer
MODEL = "cardiffnlp/twitter-roberta-base-sentiment"
tokenizer = AutoTokenizer.from_pretrained(MODEL, from_tf=True)
model = AutoModelForSequenceClassification.from_pretrained(MODEL)


In [22]:
labels = ['negative', 'neutral', 'positive']


In [23]:
def get_sentiment(text):
    # Tokenize and encode text
    encoded_input = tokenizer(text, return_tensors='pt')
    with torch.no_grad():
        output = model(**encoded_input)
    
    # Convert logits to probabilities
    scores = output.logits[0].numpy()
    probs = torch.nn.functional.softmax(torch.tensor(scores), dim=0)
    
    # Get the label with highest probability
    top_class = torch.argmax(probs).item()
    
    return {
        'label': labels[top_class],
        'confidence': round(probs[top_class].item(), 3),
        'all_scores': dict(zip(labels, [round(p.item(), 3) for p in probs]))
    }


In [24]:
sample_text = "Breaking: President sacks entire cabinet amid economic crash"
get_sentiment(sample_text)


{'label': 'negative',
 'confidence': 0.756,
 'all_scores': {'negative': 0.756, 'neutral': 0.237, 'positive': 0.007}}

In [26]:
emotion_model_name = "bhadresh-savani/distilbert-base-uncased-emotion"
emotion_tokenizer = AutoTokenizer.from_pretrained(emotion_model_name)
emotion_model = AutoModelForSequenceClassification.from_pretrained(emotion_model_name)


tokenizer_config.json:   0%|          | 0.00/291 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/768 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling back to regular HTTP download. For better performance, install the package with: `pip install huggingface_hub[hf_xet]` or `pip install hf_xet`


model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

In [30]:
emotion_labels = [
    'admiration', 'amusement', 'anger', 'annoyance', 'approval', 'caring',
    'confusion', 'curiosity', 'desire', 'disappointment', 'disapproval',
    'disgust', 'embarrassment', 'excitement', 'fear', 'gratitude', 'grief',
    'joy', 'love', 'nervousness', 'optimism', 'pride', 'realization',
    'relief', 'remorse', 'sadness', 'surprise', 'neutral'
]

In [28]:
def classify_emotion(text):
    inputs = emotion_tokenizer(text, return_tensors="pt", truncation=True)
    
    with torch.no_grad():
        outputs = emotion_model(**inputs)
        scores = F.softmax(outputs.logits, dim=1)
    
    predicted_class = torch.argmax(scores).item()
    confidence = torch.max(scores).item()

    return {
        "label": emotion_labels[predicted_class],
        "confidence": round(confidence * 100, 2)
    }


In [34]:
text = "I'm extremely undisturbed by the state of the country"
result = classify_emotion(text)
print(f"Emotion: {result['label']} ({result['confidence']}%)")

Emotion: approval (94.01%)


In [35]:
# Load tokenizer and model
model_name = "unitary/toxic-bert"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSequenceClassification.from_pretrained(model_name)

# Create pipeline
bias_classifier = pipeline("text-classification", model=model, tokenizer=tokenizer)

tokenizer_config.json:   0%|          | 0.00/174 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/811 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/438M [00:00<?, ?B/s]

Device set to use cpu


In [36]:
text = "These corrupt politicians are destroying our country again."

result = bias_classifier(text)
print(result)


[{'label': 'toxic', 'score': 0.7388682961463928}]


In [37]:
def detect_bias(text, threshold=0.7):
    result = bias_classifier(text)[0]
    if result['score'] >= threshold and result['label'] == 'TOXIC':
        return "Biased or Manipulative (TOXIC)"
    else:
        return "Likely Neutral"

In [38]:
sample_text = "They want us to believe lies — this regime is the worst ever."
print(detect_bias(sample_text))

Likely Neutral


In [41]:
NIGERIAN_TRIGGER_PHRASES = [
    r"breaking[:\-]?", r"shocking truth", r"aswear", r"they don’t want you to know",
    r"na them", r"this country is finished", r"government is lying", r"our leaders have failed us",
    r"dem wan kill us", r"nawa o", r"wetin be this", r"shey na joke", r"this is madness"
]

def has_misleading_pattern(text):
    text = text.lower()
    for pattern in NIGERIAN_TRIGGER_PHRASES:
        if re.search(pattern, text):
            return True
    return False


In [46]:
def detect_fake_news(text):
    fake_patterns = [
        r"\bBreaking\b", r"\bShocking\b", r"\bRevealed\b", r"\bExposed\b",
        r"\bScandal\b", r"what they (don’t|don't) want you to know",
        r"\bCollapse\b", r"\bTotal failure\b", r"Wake up Nigeria", r"\bMust watch\b",
        r"\bAgenda\b", r"\bHidden truth\b"
    ]
    
    text_lower = text.lower()
    match_found = False
    matched_phrases = []

    for pattern in fake_patterns:
        matches = re.findall(pattern, text, re.IGNORECASE)
        if matches:
            match_found = True
            matched_phrases.extend(matches)

    return match_found, matched_phrases


In [50]:
def get_trust_indicator(score):
    if score >= 70:
        return "🟢 Trusted"
    elif 40 <= score < 70:
        return "🟡 Caution"
    else:
        return "🔴 Risky"

def calculate_trust_score(bias_score, emotion_score, sentiment, text):
    # Flag checks
    nigerian_flag = has_misleading_pattern(text)
    fake_flag, fake_phrases = detect_fake_news(text)

    # === WEIGHTS ===
    w_bias = 0.3
    w_emotion = 0.25
    w_sentiment = 0.15
    w_nigeria = 0.2
    w_fake = 0.1

    # === SENTIMENT SCORING ===
    sentiment_penalty = {
        "negative": 1.0,  # max risk
        "neutral": 0.5,
        "positive": 0.2   # low risk
    }.get(sentiment.lower(), 0.5)

    # === MAIN SCORE CALCULATION ===
    risk_score = (
        bias_score * w_bias +
        emotion_score * w_emotion +
        sentiment_penalty * w_sentiment +
        (1 if nigerian_flag else 0) * w_nigeria +
        (1 if fake_flag else 0) * w_fake
    )

    score = max(0, min(100, 100 * (1 - risk_score)))  # Normalize trust score

    # === EXPLANATIONS ===
    explanation = []
    if bias_score > 0.6:
        explanation.append("Text contains biased or one-sided language.")
    if emotion_score > 0.6:
        explanation.append("Text is emotionally charged.")
    if sentiment.lower() == 'negative':
        explanation.append("Tone is negative.")
    if nigerian_flag:
        explanation.append("Contains potentially manipulative Nigerian expressions.")
    if fake_flag:
        explanation.append(f"Text contains suspicious or misleading phrases like: {', '.join(set(fake_phrases))}")

    indicator = get_trust_indicator(score)

    return score, indicator, explanation


In [51]:
text = "BREAKING: I’m tired of this government lying to us every day!"
bias_score = 0.7
emotion_score = 0.8
sentiment = "negative"

score, indicator, explanation = calculate_trust_score(bias_score, emotion_score, sentiment, text)

print("Trust Score:", score)
print("Indicator:", indicator)
print("Why flagged:")
for reason in explanation:
    print("-", reason)


Trust Score: 14.000000000000002
Indicator: 🔴 Risky
Why flagged:
- Text contains biased or one-sided language.
- Text is emotionally charged.
- Tone is negative.
- Contains potentially manipulative Nigerian expressions.
- Text contains suspicious or misleading phrases like: BREAKING
